# A. Biodata Mahasiswa
## Mata Kuliah: Dasar Ilmu Data (GIK2GAB3)
## Materi : Model K-Nearest Neighbors (KNN) & Hyperparameter Tuning pada studi kasus Klasifikasi menggunakan dataset Bank Marketing

*   NIM   : 707012400023
*   Nama  : Ghazy Nabil Alghifari
*   Kelas : 48-03

**Studi Kasus:** Memprediksi apakah seorang nasabah bank akan membuka rekening deposito berjangka (`deposit` = yes/no) setelah dihubungi melalui campaign marketing telepon.

Sumber dataset: [Bank Marketing Dataset – UCI / Kaggle](https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset)

# B. Load Library dan Dataset

## B1. Load library

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import pickle
import warnings
warnings.filterwarnings('ignore')

## B2. Load dataset

In [ ]:
# Load dataset
df = pd.read_csv('archive/bank.csv')

# Tampilkan ukuran dataset
print(f"Ukuran dataset: {df.shape[0]} baris, {df.shape[1]} kolom")

# Tampilkan data
df

In [ ]:
df.info()

# C. Klasifikasi dengan K-Nearest Neighbors (KNN)
Model yang digunakan adalah **KNeighborsClassifier** dari scikit-learn.

## C1. Tahap 1: Preprocessing Data

Sebelum menentukan fitur dan label, kita perlu melakukan preprocessing:
1. **Handling Outliers** menggunakan IQR capping agar nilai ekstrim tidak mempengaruhi model.
2. **Encoding** variabel kategorikal menjadi numerik.
3. **Feature Scaling** menggunakan StandardScaler agar semua fitur berada pada skala yang sama (sangat penting untuk KNN karena menggunakan jarak antar titik data).

In [ ]:
# ── 1a. Capping Outliers menggunakan metode IQR ────────────────────────────
df_clean = df.copy()

outlier_cols = ['balance', 'duration', 'campaign', 'pdays', 'previous']
for col in outlier_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)

print("✅ Outliers berhasil di-cap menggunakan nilai batas IQR.")
df_clean[outlier_cols].describe()

In [ ]:
# ── 1b. Encoding ────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

# Encode Target Variable (deposit: yes -> 1, no -> 0)
df_clean['deposit'] = df_clean['deposit'].map({'yes': 1, 'no': 0})

# Encode Binary Categorical Columns
binary_cols = ['default', 'housing', 'loan']
le = LabelEncoder()
for col in binary_cols:
    df_clean[col] = le.fit_transform(df_clean[col])

# One-Hot Encoding untuk kolom nominal multi-kelas
nominal_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
df_preprocessed = pd.get_dummies(df_clean, columns=nominal_cols, drop_first=True)

# Konversi boolean ke int
bool_cols = df_preprocessed.select_dtypes(include=['bool']).columns
df_preprocessed[bool_cols] = df_preprocessed[bool_cols].astype(int)

print(f"✅ Ukuran data setelah preprocessing encoding: {df_preprocessed.shape}")
df_preprocessed.head()

## C2. Tahap 2: Tentukan fitur dan label

- **Label (y)**: kolom `deposit` (0 = tidak membuka deposito, 1 = membuka deposito)
- **Fitur (X)**: semua kolom lain

In [ ]:
# Pisahkan Fitur (X) dan Target (y)
label_df = df_preprocessed['deposit']
fitur_df = df_preprocessed.drop('deposit', axis=1)

# Tampilkan labelnya
print("Label (deposit):")
print(label_df.value_counts())

In [ ]:
# Tampilkan fiturnya
print(f"Jumlah fitur: {fitur_df.shape[1]} kolom")
fitur_df

## C3. Tahap 3: Bagi dataset menjadi data latih dan data uji

Sebelum melakukan training, dataset perlu dibagi ke dalam X_train, X_test, Y_train, dan Y_test:
- **X_train**: Data fitur untuk **melatih** model
- **X_test**: Data fitur untuk **menguji** model setelah dilatih
- **Y_train**: Data label yang cocok dengan data di X_train
- **Y_test**: Data label yang cocok dengan data di X_test

Pembagian dataset: **80% data latih** dan **20% data uji**.

*Dataset training selalu lebih banyak dibandingkan dengan dataset testing.*

In [ ]:
from sklearn.model_selection import train_test_split

# Bagi data dengan rasio 80% latih dan 20% uji, stratify agar proporsi kelas seimbang
X_train, X_test, Y_train, Y_test = train_test_split(
    fitur_df, label_df, test_size=0.2, random_state=42, stratify=label_df
)

print("Banyak data latih (fitur) setelah dilakukan Train-Test Split: ", len(X_train))
print("Banyak data latih (label) setelah dilakukan Train-Test Split: ", len(Y_train))
print("Banyak data uji (fitur) setelah dilakukan Train-Test Split:   ", len(X_test))
print("Banyak data uji (label) setelah dilakukan Train-Test Split:   ", len(Y_test))

In [ ]:
# ── Feature Scaling menggunakan StandardScaler ──────────────────────────────
# KNN sangat sensitif terhadap skala fitur karena menggunakan perhitungan jarak,
# sehingga scaling wajib dilakukan
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numerical_features = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test_scaled[numerical_features]  = scaler.transform(X_test[numerical_features])

print("✅ Standard Scaling berhasil diterapkan pada kolom numerik.")
X_train_scaled[numerical_features].head()

## C4. Tahap 4: Siapkan classifier dan tentukan variabel/parameternya

Model yang digunakan adalah **KNeighborsClassifier** (K-Nearest Neighbors).

Referensi:
- https://scikit-learn.org/stable/modules/neighbors.html
- https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Variable classifier untuk model KNN tanpa HPO
# Menggunakan parameter default: K=5, weights='uniform', metric='minkowski'
classifier = KNeighborsClassifier(n_neighbors=5)

print("✅ Classifier KNN berhasil diinisialisasi.")
print(classifier)

## C5. Tahap 5: Lakukan proses training dengan data latih

In [ ]:
# X_train_scaled = data fitur latih (sudah di-scaling)
# Y_train         = data label latih
classifier.fit(X_train_scaled, Y_train)

print("✅ Training model KNN selesai!")

## C6. Tahap 6: Lakukan pengujian dengan data uji

In [ ]:
# X_test_scaled = data fitur uji (sudah di-scaling)
hasil_klasifikasi = classifier.predict(X_test_scaled)

In [ ]:
# Lihat hasil prediksinya
print("Hasil prediksi (5 data pertama):")
print(hasil_klasifikasi[:5])

In [ ]:
# Bandingkan hasil prediksi dengan label yang sesungguhnya
df_hasil_klasifikasi = pd.DataFrame({
    "prediksi"   : hasil_klasifikasi,
    "label_asli" : Y_test.reset_index(drop=True)
})

df_hasil_klasifikasi

In [ ]:
# Lihat data prediksi yang berbeda dengan label asli
data_salah = df_hasil_klasifikasi[df_hasil_klasifikasi['prediksi'] != df_hasil_klasifikasi['label_asli']]
print(f"Jumlah data yang diprediksi salah: {len(data_salah)}")

In [ ]:
# Mengetahui jumlah prediksi yang benar (True) dan salah (False)
perbandingan = df_hasil_klasifikasi['prediksi'] == df_hasil_klasifikasi['label_asli']
print(perbandingan.value_counts())

In [ ]:
# Simpan fitur, prediksi, dan label asli dari data uji
hasil_lengkap = pd.concat(
    [X_test.reset_index(drop=True),
     pd.DataFrame({
         'prediksi'  : hasil_klasifikasi,
         'label_asli': Y_test.reset_index(drop=True)
     })],
    axis=1
)

hasil_lengkap

In [ ]:
# Simpan hasil klasifikasi ke CSV
hasil_lengkap.to_csv('hasilKlasifikasiKNN_BankMarketing.csv', index=False)
print("✅ File hasilKlasifikasiKNN_BankMarketing.csv berhasil disimpan.")

In [ ]:
# Simpan model menggunakan joblib
joblib.dump(classifier, 'modelJb_KNN.joblib')
print("✅ Model disimpan ke modelJb_KNN.joblib")

In [ ]:
# Simpan model menggunakan pickle
with open('modelPK_KNN.pkl', 'wb') as file:
    pickle.dump(classifier, file)
print("✅ Model disimpan ke modelPK_KNN.pkl")

## C7. Tahap 7: Analisa Performansi Model

### C7.1 Menggunakan accuracy score

In [ ]:
# Hitung akurasi skor
akurasi = classifier.score(X_test_scaled, Y_test)
print(f"Akurasi Model KNN (tanpa HPO): {akurasi:.4f} ({akurasi*100:.2f}%)")

Accuracy score menunjukkan persentase data uji yang berhasil diprediksi dengan benar oleh model.
Nilai yang mendekati 1,0 (100%) menunjukkan model memiliki kemampuan prediksi yang baik.
Meskipun demikian, evaluasi tambahan seperti confusion matrix dan classification report tetap diperlukan,
terutama untuk mengetahui performa model pada masing-masing kelas.

### C7.2 Menggunakan confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix

# Hitung confusion matrix
cm = confusion_matrix(Y_test, hasil_klasifikasi)

# Plot
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Deposit (0)', 'Deposit (1)'],
            yticklabels=['No Deposit (0)', 'Deposit (1)'])

plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix – KNN Tanpa HPO')
plt.tight_layout()
plt.show()

# Detail nilai confusion matrix
tn, fp, fn, tp = cm.ravel()
print(f"True Negative  (TN) = {tn}")
print(f"False Positive (FP) = {fp}")
print(f"False Negative (FN) = {fn}")
print(f"True Positive  (TP) = {tp}")
print(f"\nTotal prediksi benar = {tn + tp}")
print(f"Total prediksi salah = {fp + fn}")

Confusion matrix membandingkan hasil prediksi model dengan label sebenarnya:
- **True Negative (TN)**: Data *No Deposit* yang diprediksi benar sebagai *No Deposit*.
- **False Positive (FP)**: Data *No Deposit* yang salah diprediksi sebagai *Deposit*.
- **False Negative (FN)**: Data *Deposit* yang salah diprediksi sebagai *No Deposit*.
- **True Positive (TP)**: Data *Deposit* yang diprediksi benar sebagai *Deposit*.

Semakin besar nilai TN dan TP (diagonal utama), semakin baik performa model.

### C7.3 Menggunakan classification report

In [ ]:
from sklearn.metrics import classification_report

# Tampilkan classification report
print(classification_report(Y_test, hasil_klasifikasi, target_names=['No Deposit', 'Deposit']))

Classification report menampilkan metrik evaluasi yang lebih detail untuk setiap kelas:

- **Precision**: Dari semua data yang diprediksi sebagai kelas tertentu, berapa persen yang benar?
- **Recall**: Dari semua data yang sebenarnya termasuk kelas tertentu, berapa persen yang berhasil dikenali model?
- **F1-Score**: Rata-rata harmonik dari precision dan recall. Berguna ketika distribusi kelas tidak seimbang.
- **Support**: Jumlah data aktual pada setiap kelas.
- **Accuracy**: Persentase total prediksi yang benar dari seluruh data uji.

# D. Kesimpulan Umum Tanpa Menggunakan HPO

Berdasarkan hasil evaluasi performa model menggunakan accuracy score, confusion matrix, dan classification report,
model klasifikasi K-Nearest Neighbors (KNN) dengan KNeighborsClassifier menunjukkan kemampuan yang baik dalam memprediksi
apakah seorang nasabah bank akan membuka rekening deposito berjangka, **sebelum** dilakukan Hyperparameter Optimization (HPO).

**Ringkasan hasil evaluasi:**

| Metrik | Nilai |
|--------|-------|
| Accuracy Score | (lihat output sel C7.1) |
| Confusion Matrix | (lihat output sel C7.2) |
| Classification Report | (lihat output sel C7.3) |

Hasil di atas menunjukkan performa baseline model KNN dengan parameter default (K=5).
Selanjutnya, akan dilakukan optimasi hyperparameter menggunakan GridSearchCV untuk mencari kombinasi parameter terbaik.

# E. Optimasi: Hyperparameter Tuning dengan GridSearch

## E1. Tahap 1: Pencarian parameter yang optimal

GridSearchCV akan mencoba semua kombinasi parameter yang telah ditentukan menggunakan
Cross Validation untuk menemukan kombinasi terbaik.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Tentukan parameter apa saja yang ingin dicoba dan nilainya
param_grid = [
    {
        'n_neighbors'  : [3, 5, 7, 9, 11, 13, 15, 17, 19, 21],
        'weights'      : ['uniform', 'distance'],
        'metric'       : ['euclidean', 'manhattan']
    }
]

In [ ]:
# Siapkan Grid Search dengan 5-fold cross validation
# n_jobs=-1: Semua core CPU akan digunakan untuk mempercepat proses
grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    scoring='accuracy',
    cv=5,
    verbose=1,
    n_jobs=-1
)

In [ ]:
# Latih model dengan GridSearchCV
# Catatan: Proses ini bisa memakan waktu beberapa menit karena mencoba banyak kombinasi parameter
import time
start = time.time()
grid.fit(X_train_scaled, Y_train)
elapsed = time.time() - start

print(f"\n✅ GridSearchCV selesai dalam {elapsed:.1f} detik")

In [ ]:
# Tampilkan kombinasi parameter terbaik
print("Best Parameters hasil GridSearchCV:")
print(grid.best_params_)
print(f"\nBest CV Accuracy: {grid.best_score_ * 100:.2f}%")

## E2. Tahap 2: Uji coba optimasi model KNN menggunakan parameter hasil HPO

### E2.1 Siapkan variabel classifier dan tentukan parameternya berdasarkan hasil HPO

In [ ]:
# Gunakan parameter terbaik dari hasil GridSearchCV
best_params = grid.best_params_

# Variable classifier2 untuk model KNN dengan parameter HPO
classifier2 = KNeighborsClassifier(
    n_neighbors = best_params['n_neighbors'],
    weights     = best_params['weights'],
    metric      = best_params['metric']
)

print("✅ Classifier KNN dengan parameter HPO berhasil diinisialisasi.")
print(classifier2)

### E2.2 Lakukan proses training dengan data latih

In [ ]:
# X_train_scaled = data fitur latih
# Y_train         = data label latih
classifier2.fit(X_train_scaled, Y_train)

print("✅ Training model HPO selesai!")

### E2.3 Lakukan pengujian dengan data uji

In [ ]:
# X_test_scaled = data fitur uji
hasil_klasifikasi2 = classifier2.predict(X_test_scaled)

In [ ]:
# Lihat hasil prediksinya
print("Hasil prediksi HPO (5 data pertama):")
print(hasil_klasifikasi2[:5])

In [ ]:
# Bandingkan hasil prediksi dengan label yang sesungguhnya
df_hasil_klasifikasi2 = pd.DataFrame({
    "prediksi"   : hasil_klasifikasi2,
    "label_asli" : Y_test.reset_index(drop=True)
})

df_hasil_klasifikasi2

In [ ]:
# Data yang prediksi dan label aslinya tidak sama
data_salah2 = df_hasil_klasifikasi2[df_hasil_klasifikasi2['prediksi'] != df_hasil_klasifikasi2['label_asli']]
print(f"Jumlah data yang diprediksi salah (HPO): {len(data_salah2)}")

In [ ]:
# Simpan fitur, prediksi dan label asli dari data uji
hasil_lengkap2 = pd.concat(
    [X_test.reset_index(drop=True),
     pd.DataFrame({
         'prediksi'  : hasil_klasifikasi2,
         'label_asli': Y_test.reset_index(drop=True)
     })],
    axis=1
)

hasil_lengkap2

In [ ]:
# Simpan hasil klasifikasi HPO ke CSV
hasil_lengkap2.to_csv('hasilKlasifikasiKNN_HPO_BankMarketing.csv', index=False)
print("✅ File hasilKlasifikasiKNN_HPO_BankMarketing.csv berhasil disimpan.")

In [ ]:
# Simpan model HPO menggunakan joblib
joblib.dump(classifier2, 'modelJb_KNN-HPO.joblib')
print("✅ Model HPO disimpan ke modelJb_KNN-HPO.joblib")

In [ ]:
# Simpan model HPO menggunakan pickle
with open('modelPK_KNN-HPO.pkl', 'wb') as file:
    pickle.dump(classifier2, file)
print("✅ Model HPO disimpan ke modelPK_KNN-HPO.pkl")

## E3. Tahap 3: Analisa Performansi Model (Setelah HPO)

### E3.1 Menggunakan accuracy score

In [ ]:
# Hitung akurasi skor model HPO
akurasi2 = classifier2.score(X_test_scaled, Y_test)
print(f"Akurasi Model KNN (dengan HPO): {akurasi2:.4f} ({akurasi2*100:.2f}%)")

Accuracy score pada model setelah HPO menunjukkan apakah proses optimasi parameter berhasil
meningkatkan kemampuan model dalam memprediksi data uji dibandingkan model tanpa HPO.

### E3.2 Menggunakan confusion matrix

In [ ]:
# Hitung confusion matrix model HPO
cm2 = confusion_matrix(Y_test, hasil_klasifikasi2)

# Plot
plt.figure(figsize=(6, 4))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['No Deposit (0)', 'Deposit (1)'],
            yticklabels=['No Deposit (0)', 'Deposit (1)'])

plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix – KNN Dengan HPO')
plt.tight_layout()
plt.show()

# Detail nilai confusion matrix
tn2, fp2, fn2, tp2 = cm2.ravel()
print(f"True Negative  (TN) = {tn2}")
print(f"False Positive (FP) = {fp2}")
print(f"False Negative (FN) = {fn2}")
print(f"True Positive  (TP) = {tp2}")
print(f"\nTotal prediksi benar = {tn2 + tp2}")
print(f"Total prediksi salah = {fp2 + fn2}")

Confusion matrix setelah HPO dibandingkan dengan sebelum HPO untuk melihat apakah
terjadi peningkatan pada jumlah prediksi yang benar dan penurunan pada jumlah prediksi yang salah.

### E3.3 Menggunakan classification report

In [ ]:
# Tampilkan classification report model HPO
print(classification_report(Y_test, hasil_klasifikasi2, target_names=['No Deposit', 'Deposit']))

Classification report setelah HPO menampilkan nilai precision, recall, f1-score, dan accuracy
untuk masing-masing kelas. Perbandingan dengan hasil sebelum HPO memberikan gambaran nyata
tentang dampak dari optimasi hyperparameter.

# F. Kesimpulan Umum Penggunaan Parameter HPO

Berdasarkan accuracy score, confusion matrix, dan classification report, berikut perbandingan
performa model K-Nearest Neighbors (KNN) **sebelum** dan **sesudah** Hyperparameter Optimization (HPO):

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Metrik model tanpa HPO
acc1   = accuracy_score(Y_test, hasil_klasifikasi)
prec1  = precision_score(Y_test, hasil_klasifikasi, average='weighted')
rec1   = recall_score(Y_test, hasil_klasifikasi, average='weighted')
f11    = f1_score(Y_test, hasil_klasifikasi, average='weighted')

# Metrik model dengan HPO
acc2   = accuracy_score(Y_test, hasil_klasifikasi2)
prec2  = precision_score(Y_test, hasil_klasifikasi2, average='weighted')
rec2   = recall_score(Y_test, hasil_klasifikasi2, average='weighted')
f12    = f1_score(Y_test, hasil_klasifikasi2, average='weighted')

# Tabel perbandingan
tabel = pd.DataFrame({
    'Metrik'    : ['Accuracy', 'Precision (weighted)', 'Recall (weighted)', 'F1-Score (weighted)'],
    'Tanpa HPO' : [f"{acc1:.4f}", f"{prec1:.4f}", f"{rec1:.4f}", f"{f11:.4f}"],
    'Dengan HPO': [f"{acc2:.4f}", f"{prec2:.4f}", f"{rec2:.4f}", f"{f12:.4f}"],
    'Selisih'   : [f"{(acc2-acc1)*100:+.2f}%", f"{(prec2-prec1)*100:+.2f}%",
                   f"{(rec2-rec1)*100:+.2f}%",  f"{(f12-f11)*100:+.2f}%"]
})

print("=" * 65)
print("  PERBANDINGAN PERFORMA MODEL KNN: TANPA HPO vs DENGAN HPO")
print("=" * 65)
print(tabel.to_string(index=False))
print("=" * 65)

In [ ]:
# Visualisasi perbandingan Confusion Matrix (berdampingan)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, cm_data, title, cmap in zip(
    axes,
    [cm, cm2],
    ['Tanpa HPO', 'Dengan HPO'],
    ['Blues', 'Oranges']
):
    sns.heatmap(cm_data, annot=True, fmt='d', cmap=cmap,
                xticklabels=['No Deposit', 'Deposit'],
                yticklabels=['No Deposit', 'Deposit'],
                ax=ax)
    ax.set_title(f'Confusion Matrix – {title}')
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Aktual')

plt.suptitle('Perbandingan Confusion Matrix: Tanpa HPO vs Dengan HPO', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Kesimpulan:**

Berdasarkan perbandingan hasil evaluasi sebelum dan sesudah HPO, dapat disimpulkan:

1. **Accuracy Score**: Bandingkan nilai akurasi sebelum dan sesudah HPO. Jika meningkat, berarti HPO berhasil mengoptimalkan model.

2. **Confusion Matrix**: Perhatikan perubahan pada jumlah prediksi benar (TN + TP) dan prediksi salah (FP + FN) sebelum dan sesudah HPO.

3. **Classification Report**: Evaluasi perubahan nilai precision, recall, dan f1-score pada kedua kelas (*No Deposit* dan *Deposit*).

4. **Tabel Perbandingan**: Lihat selisih metrik pada tabel di atas untuk mengetahui dampak kuantitatif dari HPO.

Secara keseluruhan, proses Hyperparameter Optimization menggunakan GridSearchCV membantu menemukan kombinasi parameter optimal
(n_neighbors, weights, metric) sehingga performa model KNN dapat ditingkatkan dibandingkan model baseline.